# Lesson 6.8: Effect Size and Practical Significance

Compute Cohen's d for the memory-training study by hand, then see a huge sample make a tiny effect significant while d stays tiny.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A school tests a memory-training program with 25 trained students and 25 untrained students on the same memory test. The trained group averages 78 and the control group 70, and the t-test is significant. The principal asks a harder question: how BIG is the improvement in a way that does not depend on the sample size, and would a tiny improvement also look 'significant' in a very large study?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the two groups

H0: the true average memory scores of trained and control students are equal. H1: they differ. The scores below are built to match the lesson: trained mean 78 (SD 10), control mean 70 (SD 12), 25 students per group.

In [1]:
import numpy as np  # Load NumPy for arrays and square roots, under the short name np.
from scipy import stats  # Load SciPy's statistics tools for the t-test.

trained = np.array([95.69, 56.78, 81.95, 73.61, 74.58, 76.59, 61.32, 76.45, 71.09, 106.54, 80.32, 75.43, 76.03, 72.76, 69.48, 75.11, 82.49, 76.39, 86.52, 76.72, 78.62, 91.5, 83.03, 74.14, 76.87])  # Store the 25 scores of the trained students.
control = np.array([62.92, 68.16, 88.37, 77.33, 52.05, 70.03, 63.23, 71.72, 52.41, 72.74, 72.67, 87.4, 73.57, 75.7, 53.68, 94.84, 49.03, 82.19, 66.46, 60.41, 62.87, 62.7, 74.26, 68.88, 86.38])  # Store the 25 scores of the control students.

for name, g in [('trained', trained), ('control', control)]:  # Loop over both groups with their names, so one piece of code describes each group.
    print(f'{name}: n = {len(g)}, mean = {g.mean():.2f}, sd = {g.std(ddof=1):.2f}')  # Print each group's size, mean and sample SD (ddof=1). We always look at the numbers before testing them.

trained: n = 25, mean = 78.00, sd = 10.00
control: n = 25, mean = 70.00, sd = 12.00


**What this means.** The trained group averages 78.00 (SD 10.00) and the control group 70.00 (SD 12.00), with 25 students each. The raw gap is 8 points, but 8 points means little until we know how spread out scores are. That is what Cohen's d measures.

## Step 2: Cohen's d by hand

Cohen's d is the gap between the means measured in pooled standard deviations. The pooled SD is a weighted average of the two group SDs, so d does not depend on the sample size.

In [2]:
n1, n2 = len(trained), len(control)  # Store the two group sizes so the formulas below stay short.
s1, s2 = trained.std(ddof=1), control.std(ddof=1)  # Store the two sample SDs; the pooled SD will blend them.

sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))  # The pooled SD: a weighted average of the two group variances (weights n - 1), then a square root. It is the typical spread in either group.
d = (trained.mean() - control.mean()) / sp  # Cohen's d: the gap between the means divided by the pooled SD. It says how many standard deviations apart the groups are.
g = d * (1 - 3 / (4 * (n1 + n2 - 2) - 1))   # Hedges' g, a small-sample correction  # Hedges' g shrinks d slightly to correct a small-sample bias. For small groups it is a little more honest.

print(f'pooled SD = {sp:.3f}')  # Print the pooled SD.
print(f"Cohen's d = {d:.3f}")  # Print Cohen's d to 3 decimals.
print(f"Hedges' g = {g:.3f}")  # Print Hedges' g to 3 decimals.

pooled SD = 11.045
Cohen's d = 0.724
Hedges' g = 0.713


**What this means.** The pooled SD is 11.045, so the 8-point gap is d = 0.724 pooled SDs: a medium-to-large effect by the usual benchmarks (0.2 small, 0.5 medium, 0.8 large). Hedges' g is slightly smaller, 0.713. Unlike a p-value, d does not grow just because we collect more students.

## Step 3: Add the p-value

The t-test says whether the gap could be chance, while d says how big it is. Report both. (This is the pooled test; the lesson's Welch version gives the same p to three decimals.)

In [3]:
res = stats.ttest_ind(trained, control)  # Run the pooled two-sample t-test to ask whether a gap this big could be chance.

print(f't = {res.statistic:.3f}, df = {res.df:.0f}, p = {res.pvalue:.4f}')  # Print t, df and the p-value together, so we can report significance next to the effect size.

t = 2.561, df = 48, p = 0.0136


**What this means.** t = 2.561, df = 48, p = 0.0136. If the program did nothing, a gap this large would appear about 1.4% of the time, so it is statistically significant at 0.05. The p-value says the gap is probably not zero; d = 0.724 says it is a sizeable gap. Always report both.

## Step 4: Tiny effect, huge sample

ttest_ind_from_stats runs the t-test from summary numbers. Compare three studies: a huge sample with a trivial gap, the memory study, and a tiny sample with a big gap. The p-values and d tell different stories.

In [4]:
def study(label, diff, sd, n):  # Define a helper that runs a t-test from summary numbers and prints d, t and p for a study.
    res = stats.ttest_ind_from_stats(diff, sd, n, 0, sd, n)   # pooled test  # ttest_ind_from_stats needs only mean, SD and size of each group. Group 2 has mean 0, so the first mean is the gap.
    print(f'{label:<14} d = {diff / sd:.3f}  t = {res.statistic:.2f}  p = {res.pvalue:.4f}')  # Print a one-line summary: d is diff / sd, then the test's t and p.

study('huge sample', 0.2, 15, 50000)  # Study 1: a trivial 0.2-point gap, SD 15, but 50,000 people per group.
study('memory study', 8, sp, 25)  # Study 2: our memory study, an 8-point gap with the pooled SD and 25 per group.
study('tiny sample', 15, 15, 4)  # Study 3: a big 15-point gap (d = 1) but only 4 people per group.

huge sample    d = 0.013  t = 2.11  p = 0.0350
memory study   d = 0.724  t = 2.56  p = 0.0136
tiny sample    d = 1.000  t = 1.41  p = 0.2070


**What this means.** The huge study has d = 0.013, trivially small, yet p = 0.0350 because 50,000 people can detect almost anything. The tiny study has d = 1.000, a big effect, yet p = 0.2070, not significant, because 4 people cannot. Our memory study sits in between. p reflects sample size; d does not. Significance is not importance.

## So, what is the answer?

The memory training raised scores by 8 points, which is d = 0.72 pooled SDs (Hedges' g 0.71) and statistically significant (t = 2.561, p = 0.0136). Caution: p and d answer different questions. The same p can come from a huge sample with a trivial effect (d = 0.013 gave p = 0.035) or be missed by a tiny sample with a big one, so always report the effect size too.

## Now you try

Two maths classes of 40 students each take the same test. Class 1 used a new practice app and Class 2 did not. Compute Cohen's d for the difference, run the pooled t-test, and then check what a tiny gap (0.5 points, SD 10) would give with 20,000 students per group.

**Data:** g1 = [79, 69, 68, 69, 87, 68, 69, 76, 71, 70, 61, 72, 68, 84, 79, 72, 79, 69, 83, 72, 78, 59, 75, 55, 52, 69, 63, 74, 94, 64, 66, 74, 77, 70, 70, 79, 77, 62, 71, 72]
g2 = [55, 69, 57, 76, 68, 67, 60, 65, 46, 55, 70, 45, 74, 49, 74, 58, 74, 67, 51, 78, 80, 65, 63, 64, 56, 77, 61, 65, 58, 60, 53, 79, 64, 76, 66, 59, 63, 60, 66, 62]

1. Find each class's mean and the pooled SD (the groups are equal in size).
2. Compute Cohen's d and say whether it is small, medium or large.
3. Run the pooled t-test and report t and p.
4. Use ttest_ind_from_stats for a 0.5-point gap with SD 10 and 20,000 per group, and compare its d and p.

*Hint:* With equal group sizes the pooled SD is sqrt((var1 + var2) / 2) using ddof=1. Then d = (mean1 - mean2) / pooled SD. For task 4, ttest_ind_from_stats(0.5, 10, 20000, 0, 10, 20000).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

g1 = np.array([79, 69, 68, 69, 87, 68, 69, 76, 71, 70, 61, 72, 68, 84, 79, 72, 79, 69, 83, 72, 78, 59, 75, 55, 52, 69, 63, 74, 94, 64, 66, 74, 77, 70, 70, 79, 77, 62, 71, 72])
g2 = np.array([55, 69, 57, 76, 68, 67, 60, 65, 46, 55, 70, 45, 74, 49, 74, 58, 74, 67, 51, 78, 80, 65, 63, 64, 56, 77, 61, 65, 58, 60, 53, 79, 64, 76, 66, 59, 63, 60, 66, 62])
sp = np.sqrt((g1.var(ddof=1) + g2.var(ddof=1)) / 2)
d = (g1.mean() - g2.mean()) / sp
res = stats.ttest_ind(g1, g2)
print(f'means: {g1.mean():.2f} and {g2.mean():.2f}, pooled SD = {sp:.2f}')
print(f"Cohen's d = {d:.3f}")
print(f't = {res.statistic:.3f}, p = {res.pvalue:.6f}')
big = stats.ttest_ind_from_stats(0.5, 10, 20000, 0, 10, 20000)
print(f'huge-sample study: d = 0.05, p = {big.pvalue:.1e}')

means: 71.65 and 63.88, pooled SD = 8.64
Cohen's d = 0.900
t = 4.023, p = 0.000132
huge-sample study: d = 0.05, p = 5.8e-07


**Answer.** The means are 71.65 and 63.88 with a pooled SD of 8.64, so d = 0.900, a large effect (0.8 or more). The t-test gives t = 4.023 and p = 0.000132, so the app group really scored higher. The huge study has only d = 0.05 but p is about 5.7e-07, also tiny. Equal p-values can hide very different effects, so report d next to p.

## The same question in R, SPSS and Stata

**R**

```r
trained <- c(95.69, 56.78, 81.95, 73.61, 74.58, 76.59, 61.32, 76.45, 71.09, 106.54, 80.32, 75.43, 76.03, 72.76, 69.48, 75.11, 82.49, 76.39, 86.52, 76.72, 78.62, 91.5, 83.03, 74.14, 76.87)  # Store the 25 trained scores.
control <- c(62.92, 68.16, 88.37, 77.33, 52.05, 70.03, 63.23, 71.72, 52.41, 72.74, 72.67, 87.4, 73.57, 75.7, 53.68, 94.84, 49.03, 82.19, 66.46, 60.41, 62.87, 62.7, 74.26, 68.88, 86.38)  # Store the 25 control scores.
sp <- sqrt((24 * var(trained) + 24 * var(control)) / 48)  # The pooled SD, from the two variances weighted by n - 1 = 24 each and divided by the total df, 48.
(mean(trained) - mean(control)) / sp          # Cohen's d  # Cohen's d: the difference in means divided by the pooled SD. Base R has no ready-made function for it.
t.test(trained, control, var.equal = TRUE)  # Pooled two-sample t-test (var.equal = TRUE) for the memory study; it prints t, df and p.

study <- function(diff, sd, n) {  # Define a small function that turns a gap, an SD and a group size into d, t and a two-sided p-value, for the other two studies.
  t <- diff / sqrt(2 * sd^2 / n)  # Inside it: t is the gap divided by its standard error, sqrt(2 sd^2 / n).
  c(d = diff / sd, t = t, p = 2 * pt(-abs(t), 2 * n - 2))  # Return d, t and p as one vector; p is twice the area below -|t| on a t curve with 2n - 2 df.
}  # Close the function with a brace.
study(0.2, 15, 50000)  # Study 1: a gap of 0.2, SD 15, 50,000 people per group.
study(15, 15, 4)  # Study 3: a gap of 15, SD 15, only 4 people per group.
```

How to read it: The d line should print 0.7243 and t.test shows t = 2.5609, df = 48, p-value = 0.01364. The huge study gives d = 0.0133 with p = 0.03502 (significant but trivial), the tiny one d = 1 with p = 0.2070 (large but not significant).

**SPSS** (Analyze > Compare Means > Independent-Samples T Test (version 27 and later also prints an effect-size table with Cohen's d))

```
DATA LIST FREE / label n diff sd.
BEGIN DATA
1 50000 0.2 15
2 25 8 11.0454
3 4 15 15
END DATA.
COMPUTE d = diff / sd.
COMPUTE t = diff / SQRT(2 * sd**2 / n).
COMPUTE p = 2 * CDF.T(-ABS(t), 2 * n - 2).
LIST label d t p.
```

How to read it: LIST shows d, t and p for each study (SPSS shows 2 decimals by default): study 1 d = 0.01, t = 2.11, p = 0.04; study 2 d = 0.72, t = 2.56, p = 0.01; study 3 d = 1.00, t = 1.41, p = 0.21. Small d can be significant, and large d can be non-significant.

**Stata**

```
esizei 25 78 10 25 70 12, cohensd      // Cohen's d from summary numbers
ttesti 25 78 10 25 70 12
esizei 50000 0.2 15 50000 0 15, cohensd
ttesti 50000 0.2 15 50000 0 15
```

How to read it: For the memory study find Cohen's d of about 0.72 in the esizei table and t = 2.56 with p = 0.0136 from ttesti (using the lesson's rounded numbers). For the huge sample d is about 0.013 while the two-sided p is 0.035: significant, but trivial.